In [ ]:
%pylab inline
import numpy as np
from skimage import io, color
from skimage.feature import canny
from skimage.transform import downscale_local_mean
from scipy import ndimage as ndi
from fancy.plotting import plot_image
from skimage.morphology import skeletonize
import os

In [ ]:
infile = 'test_graph_image.jpg'
rgb = io.imread(infile)

In [ ]:
# downsample the image until one of the sides is smaller than 1000px
while min(rgb.shape[:2]) >= 1000:
    rgb = downscale_local_mean(rgb, (2, 2, 1))
    print(rgb.shape)

In [ ]:
plot_image(rgb/255, figheight=5);

In [ ]:
grayscale = color.rgb2gray(rgb)

In [ ]:
plot_image(grayscale, cmap='gray', figheight=5, colorbar=True);

In [ ]:
plt.hist(grayscale.flatten(), bins=100);

In [ ]:
fg = grayscale < 75

plot_image(fg, figheight=5)

In [ ]:
# close boundaries
dilated = ndi.binary_closing(fg, iterations=3)
plot_image(dilated, figheight=5)

In [ ]:
# skeleton = skeletonize(dilated)
skeleton = mh.thin(dilated)
plot_image(skeleton, figheight=30);

In [ ]:
import mahotas as mh
from mahotas import polygon

def branchedPoints(skel):
    branch1=np.array([[2, 1, 2], [1, 1, 1], [2, 2, 2]])
    branch2=np.array([[1, 2, 1], [2, 1, 2], [1, 2, 1]])
    branch3=np.array([[1, 2, 1], [2, 1, 2], [1, 2, 2]])
    branch4=np.array([[2, 1, 2], [1, 1, 2], [2, 1, 2]])
    branch5=np.array([[1, 2, 2], [2, 1, 2], [1, 2, 1]])
    branch6=np.array([[2, 2, 2], [1, 1, 1], [2, 1, 2]])
    branch7=np.array([[2, 2, 1], [2, 1, 2], [1, 2, 1]])
    branch8=np.array([[2, 1, 2], [2, 1, 1], [2, 1, 2]])
    branch9=np.array([[1, 2, 1], [2, 1, 2], [2, 2, 1]])
    br1=mh.morph.hitmiss(skel,branch1)
    br2=mh.morph.hitmiss(skel,branch2)
    br3=mh.morph.hitmiss(skel,branch3)
    br4=mh.morph.hitmiss(skel,branch4)
    br5=mh.morph.hitmiss(skel,branch5)
    br6=mh.morph.hitmiss(skel,branch6)
    br7=mh.morph.hitmiss(skel,branch7)
    br8=mh.morph.hitmiss(skel,branch8)
    br9=mh.morph.hitmiss(skel,branch9)
    return br1+br2+br3+br4+br5+br6+br7+br8+br9

def endPoints(skel):
    endpoint1=np.array([[0, 0, 0],
                        [0, 1, 0],
                        [2, 1, 2]])
    
    endpoint2=np.array([[0, 0, 0],
                        [0, 1, 2],
                        [0, 2, 1]])
    
    endpoint3=np.array([[0, 0, 2],
                        [0, 1, 1],
                        [0, 0, 2]])
    
    endpoint4=np.array([[0, 2, 1],
                        [0, 1, 2],
                        [0, 0, 0]])
    
    endpoint5=np.array([[2, 1, 2],
                        [0, 1, 0],
                        [0, 0, 0]])
    
    endpoint6=np.array([[1, 2, 0],
                        [2, 1, 0],
                        [0, 0, 0]])
    
    endpoint7=np.array([[2, 0, 0],
                        [1, 1, 0],
                        [2, 0, 0]])
    
    endpoint8=np.array([[0, 0, 0],
                        [2, 1, 0],
                        [1, 2, 0]])
    
    ep1=mh.morph.hitmiss(skel,endpoint1)
    ep2=mh.morph.hitmiss(skel,endpoint2)
    ep3=mh.morph.hitmiss(skel,endpoint3)
    ep4=mh.morph.hitmiss(skel,endpoint4)
    ep5=mh.morph.hitmiss(skel,endpoint5)
    ep6=mh.morph.hitmiss(skel,endpoint6)
    ep7=mh.morph.hitmiss(skel,endpoint7)
    ep8=mh.morph.hitmiss(skel,endpoint8)
    ep = ep1+ep2+ep3+ep4+ep5+ep6+ep7+ep8
    return ep

def pruning(skeleton, size=None):
    '''remove iteratively end points "size" 
       times from the skeleton
    '''
    i = 0
    while True:
        i += 1
        if size is not None and i > size:
            break
        endpoints = endPoints(skeleton)
        if not endpoints.sum():
            break
        endpoints = np.logical_not(endpoints)
        skeleton = np.logical_and(skeleton,endpoints)
    return skeleton

In [ ]:
skeleton = pruning(skeleton)
branching_points = branchedPoints(skeleton) > 0

branch_point_labels, n_branch_points = ndi.label(branching_points)
edge_labels, n_edges = ndi.label(skeleton ^ branching_points)

from fancy.colors import colorize_segmentation
plot_image(colorize_segmentation(ndi.grey_dilation(edge_labels, size=3).astype(np.int32), ignore_label=0), figheight=5);

In [ ]:
# construct the graph: for every branching point, find the edges connected to it
edge_dict = {}
for bp in range(1, n_branch_points+1):
    adjacent_edges = np.unique(edge_labels[ndi.binary_dilation(branch_point_labels == bp)])
    adjacent_edges = adjacent_edges[adjacent_edges > 0]
    edge_dict[bp] = set(adjacent_edges)

In [ ]:
edges = [(bp1, bp2) 
         for (bp1, edge_ids1) in edge_dict.items()
         for (bp2, edge_ids2) in edge_dict.items()
         if bp2 > bp1 and edge_ids1.intersection(edge_ids2)
        ]

In [ ]:
# compute branching point positions
xy = np.moveaxis(np.mgrid[:skeleton.shape[0], :skeleton.shape[1]], 0, -1)[:, :, ::-1]
pos_dict = {}
for bp in range(1, n_branch_points+1):
    pos_dict[bp] = xy[branch_point_labels==bp].mean(0)

In [ ]:
lengths = [np.linalg.norm(pos_dict[i] - pos_dict[j]) for (i, j) in edges]
plt.hist(lengths, bins=20)
plt.show();

In [ ]:
from eucare.overlap import group_closeby

merge_threshold = 20
bp_positions = np.stack(list(pos_dict.values()))

new_bp_labels = group_closeby(bp_positions, merge_threshold)
print(new_bp_labels)
new_pos_dict = {int(i): np.mean([pos_dict[int(j)] for j in np.argwhere(new_bp_labels==i)+1], axis=0) for i in np.unique(new_bp_labels)}
new_edges = set((new_bp_labels[i-1], new_bp_labels[j-1]) for i, j in edges if new_bp_labels[i-1] != new_bp_labels[j-1])

In [ ]:
import networkx as nx

graph = nx.Graph()
graph.add_edges_from(new_edges)

In [ ]:
# convert to eucare graph
import eucare as ec

G = ec.conversions.EHEG_from_nx(graph, new_pos_dict)

G.recompute_lengths_and_angles()
edge_lengths = [h['length'] for h in G.halfedges_representing_edges()]
plot_image(rgb/255, figheight=5)
plt.show();
G.show()


In [ ]:
ec.io.save_graph('.'.join(os.path.basename(infile).split('.')[:-1]), G)

In [ ]:
from eucare.image_to_graph import image_to_graph

In [ ]:
G = image_to_graph('/home/roman/Downloads/2022_03_14 19_02 Office Lens (1).jpg', 
                   threshold=100, 
                   closing_iterations=5,
                   edge_length_cutoff=75)

In [ ]:
G = image_to_graph('/home/roman/Downloads/2022_03_14 19_02 Office Lens (2).jpg', 
                   threshold=100, 
                   closing_iterations=5,
                   edge_length_cutoff=35)

In [ ]:
G = image_to_graph('/home/roman/Downloads/2022_03_16 10_29 Office Lens.jpg', 
                   threshold=100, 
                   closing_iterations=4,
                   edge_length_cutoff=50)

In [ ]:
import numpy as np

def triagonalize(G):
    done=False
    while not done:
        done=True
        for f in G.faces:
            if f.order() > 3:
                done=False
                hs = [h for h in f.halfedge_iter()]
                ls = [h['length'] for h in f.halfedge_iter()]
                order = np.argsort(ls)
                # join all but 3 longest edges
                for h in np.array(hs)[order[:-3]]:
                    G.join_edge(h)



In [ ]:
from eucare.image_to_graph import image_to_graph

G = image_to_graph('/home/roman/Downloads/graph3.jpg', 
                   threshold=100, 
                   closing_iterations=8,
                   edge_length_cutoff=10,
                   max_size=250)
triagonalize(G)
G.show()

In [ ]:
import eucare as ec
ec.io.save_graph('graphs/handdrawn/egg_b.heg', G)

In [ ]:
#TODO next: merge points in triangles until everything is triangular